# Adversarial Debiasing

Train a classifier while adversarially preventing it from encoding protected attributes (city_group).

Architecture:
- **Main classifier**: BERT → profession (9 classes)
- **Adversary**: BERT embeddings → city_group (predicts protected attribute)
- **Training**: Minimize classification loss, maximize adversary loss (gradient reversal)

In [1]:
import os, json, numpy as np, pandas as pd, torch, joblib
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModel, AutoConfig
from torch.utils.data import DataLoader, Dataset as TorchDataset
from tqdm import tqdm
np.random.seed(42); torch.manual_seed(42)

/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BASE_DIR = Path("..")
df_train = pd.read_csv(BASE_DIR / "data/processed/train.csv")
df_val = pd.read_csv(BASE_DIR / "data/processed/val.csv")
df_test = pd.read_csv(BASE_DIR / "data/processed/test.csv")

mapping = pd.read_csv(BASE_DIR / "data/processed/label_to_supercategory_v1.csv")
label_to_supercat = dict(zip(mapping["label"], mapping["supercategory"]))
for df in [df_train, df_val, df_test]: 
    df["supercategory"] = df["label"].map(label_to_supercat)

# Label encoders
le_class = LabelEncoder()
df_train["y"] = le_class.fit_transform(df_train["supercategory"])
df_val["y"] = le_class.transform(df_val["supercategory"])
df_test["y"] = le_class.transform(df_test["supercategory"])
num_classes = len(le_class.classes_)

# City encoder (for adversary)
le_city = LabelEncoder()
df_train["city_id"] = le_city.fit_transform(df_train["city_group"])
df_val["city_id"] = le_city.transform(df_val["city_group"])
df_test["city_id"] = le_city.transform(df_test["city_group"])
num_cities = len(le_city.classes_)

print(f"Train: {len(df_train)}, Classes: {num_classes}, Cities: {num_cities}")

Train: 16530, Classes: 9, Cities: 41


In [3]:
# Sample weights (sqrt reweighting)
city_counts = df_train["city_group"].value_counts()
raw_w = 1.0 / np.sqrt(city_counts)
city_weight_map = (raw_w / raw_w.mean()).to_dict()
df_train["sample_weight"] = df_train["city_group"].map(city_weight_map).astype(float)
print(f"Weight range: {df_train['sample_weight'].min():.3f} - {df_train['sample_weight'].max():.3f}")

Weight range: 0.148 - 1.470


In [4]:
# Dataset
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

class ResumeDataset(TorchDataset):
    def __init__(self, df, tokenizer, max_len=128, has_weights=False):
        self.texts = df["resume_text"].values
        self.labels = df["y"].values
        self.city_ids = df["city_id"].values
        self.weights = df["sample_weight"].values if has_weights else None
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self): return len(self.texts)
    
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], padding="max_length", truncation=True, 
                             max_length=self.max_len, return_tensors="pt")
        item = {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
            "city_ids": torch.tensor(self.city_ids[idx], dtype=torch.long)
        }
        if self.weights is not None:
            item["weights"] = torch.tensor(self.weights[idx], dtype=torch.float)
        return item

train_ds = ResumeDataset(df_train, tokenizer, has_weights=True)
val_ds = ResumeDataset(df_val, tokenizer)
test_ds = ResumeDataset(df_test, tokenizer)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16)
test_loader = DataLoader(test_ds, batch_size=16)
print(f"DataLoaders ready")

DataLoaders ready


In [5]:
# Gradient Reversal Layer
class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)
    
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class GradientReversal(nn.Module):
    def __init__(self, alpha=1.0):
        super().__init__()
        self.alpha = alpha
    
    def forward(self, x):
        return GradientReversalFunction.apply(x, self.alpha)

In [6]:
# Adversarial Model
class AdversarialBERT(nn.Module):
    def __init__(self, num_classes, num_cities, adv_alpha=1.0):
        super().__init__()
        self.bert = AutoModel.from_pretrained("bert-base-uncased")
        hidden_size = self.bert.config.hidden_size  # 768
        
        # Main classifier
        self.classifier = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(hidden_size, num_classes)
        )
        
        # Adversary (predicts city from embeddings)
        self.gradient_reversal = GradientReversal(adv_alpha)
        self.adversary = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_cities)
        )
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output  # [CLS] token
        
        # Main task
        class_logits = self.classifier(pooled)
        
        # Adversary (with gradient reversal)
        reversed_pooled = self.gradient_reversal(pooled)
        city_logits = self.adversary(reversed_pooled)
        
        return class_logits, city_logits

print("AdversarialBERT defined")

AdversarialBERT defined


In [7]:
# Training config
ADV_ALPHA = 0.5  # Strength of adversarial gradient
EPOCHS = 2
LR = 2e-5

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
model = AdversarialBERT(num_classes, num_cities, adv_alpha=ADV_ALPHA).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

print(f"Device: {device}, ADV_ALPHA: {ADV_ALPHA}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 272.72it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Device: mps, ADV_ALPHA: 0.5


In [8]:
# Training loop
def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss, total_class_loss, total_adv_loss = 0, 0, 0
    correct, total = 0, 0
    
    for batch in tqdm(loader, desc="Training"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        city_ids = batch["city_ids"].to(device)
        weights = batch.get("weights")
        if weights is not None:
            weights = weights.to(device)
        
        optimizer.zero_grad()
        class_logits, city_logits = model(input_ids, attention_mask)
        
        # Classification loss (with sample weights)
        class_loss_per_sample = F.cross_entropy(class_logits, labels, reduction="none")
        if weights is not None:
            class_loss = (class_loss_per_sample * weights).mean()
        else:
            class_loss = class_loss_per_sample.mean()
        
        # Adversary loss (we want adversary to fail → model learns city-invariant features)
        adv_loss = F.cross_entropy(city_logits, city_ids)
        
        # Total loss (adversary loss is reversed by GRL, so we add it)
        loss = class_loss + adv_loss
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total_class_loss += class_loss.item()
        total_adv_loss += adv_loss.item()
        
        preds = class_logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    return {
        "loss": total_loss / len(loader),
        "class_loss": total_class_loss / len(loader),
        "adv_loss": total_adv_loss / len(loader),
        "accuracy": correct / total
    }

def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            class_logits, _ = model(input_ids, attention_mask)
            preds = class_logits.argmax(dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return np.array(all_preds), np.array(all_labels)

In [9]:
# Train
print(f"Training Adversarial Debiasing (α={ADV_ALPHA}) for {EPOCHS} epochs...")

best_val_acc = 0
for epoch in range(EPOCHS):
    metrics = train_epoch(model, train_loader, optimizer, device)
    print(f"\nEpoch {epoch+1}: loss={metrics['loss']:.4f}, class_loss={metrics['class_loss']:.4f}, "
          f"adv_loss={metrics['adv_loss']:.4f}, train_acc={metrics['accuracy']:.4f}")
    
    # Validation
    val_preds, val_labels = evaluate(model, val_loader, device)
    val_acc = accuracy_score(val_labels, val_preds)
    val_f1 = f1_score(val_labels, val_preds, average="macro")
    print(f"Val: acc={val_acc:.4f}, f1={val_f1:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "models/bert_adversarial_best.pt")
        print("  → Saved best model")

Training Adversarial Debiasing (α=0.5) for 2 epochs...


Training: 100%|██████████| 2067/2067 [40:39<00:00,  1.18s/it]



Epoch 1: loss=4.1663, class_loss=0.6788, adv_loss=3.4875, train_acc=0.4224


Evaluating: 100%|██████████| 345/345 [03:14<00:00,  1.78it/s]


Val: acc=0.5608, f1=0.5731
  → Saved best model


Training: 100%|██████████| 2067/2067 [37:54<00:00,  1.10s/it]



Epoch 2: loss=3.6203, class_loss=0.5435, adv_loss=3.0768, train_acc=0.5549


Evaluating: 100%|██████████| 345/345 [03:28<00:00,  1.65it/s]


Val: acc=0.5822, f1=0.6064
  → Saved best model


In [10]:
# Load best and evaluate on test
model.load_state_dict(torch.load("models/bert_adversarial_best.pt", weights_only=True))
y_pred, y_true = evaluate(model, test_loader, device)

test_acc = accuracy_score(y_true, y_pred)
test_f1 = f1_score(y_true, y_pred, average="macro")
print(f"\nTEST: Acc={test_acc:.4f}, F1={test_f1:.4f}")

Evaluating: 100%|██████████| 345/345 [03:29<00:00,  1.64it/s]


TEST: Acc=0.5744, F1=0.5942


In [11]:
# Fairness evaluation
df_eval = df_test.copy()
df_eval["y_true"], df_eval["y_pred"] = y_true, y_pred

def ovr_rates(df, grp, nc):
    groups = sorted(df[grp].dropna().unique())
    tpr, support = np.zeros((len(groups), nc)), np.zeros((len(groups), nc))
    for gi, g in enumerate(groups):
        dg = df[df[grp] == g]
        yt, yp = dg["y_true"].values, dg["y_pred"].values
        for c in range(nc):
            pm = (yt == c)
            TP, FN = np.sum((yp == c) & pm), np.sum((yp != c) & pm)
            support[gi, c] = pm.sum()
            tpr[gi, c] = TP / (TP + FN) if (TP + FN) > 0 else np.nan
    return tpr, support

def gaps(tpr, sup, ms=30):
    g = []
    for c in range(tpr.shape[1]):
        col = tpr[sup[:, c] >= ms, c]
        col = col[~np.isnan(col)]
        g.append(col.max() - col.min() if len(col) >= 2 else np.nan)
    g = np.array(g); v = g[~np.isnan(g)]
    return v.max() if len(v) else np.nan, v.mean() if len(v) else np.nan

tpr, sup = ovr_rates(df_eval, "city_group", num_classes)
tw, tm = gaps(tpr, sup, 30)
print(f"FAIRNESS (robust): worst={tw:.4f}, macro={tm:.4f}")
print(f"Compare: baseline=0.609 acc, 0.329 worst, 0.116 macro")

FAIRNESS (robust): worst=0.4294, macro=0.1318
Compare: baseline=0.609 acc, 0.329 worst, 0.116 macro


In [12]:
# Save model
MODEL_NAME = "bert_adversarial"
SAVE_DIR = Path("models") / MODEL_NAME
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Save full model state
torch.save(model.state_dict(), SAVE_DIR / "model.pt")

# Save BERT part for inference
model.bert.save_pretrained(SAVE_DIR / "bert")
tokenizer.save_pretrained(SAVE_DIR)

# Save classifier weights separately
torch.save(model.classifier.state_dict(), SAVE_DIR / "classifier.pt")

joblib.dump(le_class, SAVE_DIR / "label_encoder.joblib")
joblib.dump(le_city, SAVE_DIR / "city_encoder.joblib")

cfg = {
    "method": f"Adversarial Debiasing α={ADV_ALPHA} + sqrt_rw",
    "adv_alpha": ADV_ALPHA,
    "epochs": EPOCHS,
    "accuracy": float(test_acc),
    "macro_f1": float(test_f1),
    "tpr_gap_worst_robust": float(tw),
    "tpr_gap_macro_robust": float(tm),
    "num_classes": num_classes,
    "num_cities": num_cities
}
with open(SAVE_DIR / "training_config.json", "w") as f: 
    json.dump(cfg, f, indent=2)

print(f"Saved: {SAVE_DIR}")
print(f"\n{'='*50}")
print(f"SUMMARY: Adversarial Debiasing α={ADV_ALPHA}")
print(f"{'='*50}")
print(f"Accuracy: {test_acc:.4f}")
print(f"Macro F1: {test_f1:.4f}")
print(f"TPR Gap (worst): {tw:.4f}")
print(f"TPR Gap (macro): {tm:.4f}")
print(f"{'='*50}")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.26it/s]

Saved: models/bert_adversarial

SUMMARY: Adversarial Debiasing α=0.5
Accuracy: 0.5744
Macro F1: 0.5942
TPR Gap (worst): 0.4294
TPR Gap (macro): 0.1318
